# 01 — First circuit and load flow

## Objectives and engineering concept

Learn how a source, line, bus, and load become a solvable circuit, and why voltage and power units must be named.

## System, assumptions, and units

The one-line is `source — line1 — load`, three phase, 12.47 kV, 1 km. The line and load values are explicit demonstrator inputs, not measurements.

## Part A — Pure OpenDSS

The first code section repeats the manual commands, solve, and voltage extraction.

## Part B — Same study with CEPT

The typed Case expresses the same topology and assumptions.

## Part C — Compare and verify

CEPT compares the common load-bus voltage and records a verification receipt. The repeated manual setup is the pain CEPT automates; it does not replace engineering input review.

We build the same small three-phase source-line-load circuit twice: once with direct OpenDSS commands and once from a typed CEPT `Case`. The comparison is between solver-returned load-bus voltage magnitudes in per unit.

## Interpret, exercise, and reproduce

Interpret the voltage as a solver output, then change one declared line or load input and rerun all cells. The notebook displays the installed CEPT version; the final assertion is the reproducibility check.

In [ ]:
import opendssdirect as dss
from cept import Case
from cept.public import run_study

commands = [
    'Clear',
    'New Circuit.first basekv=12.47 pu=1.0 phases=3 bus1=source',
    'New Line.line1 bus1=source.1.2.3 bus2=load.1.2.3 phases=3 length=1 units=km r1=0.2 x1=0.4 r0=0.6 x0=1.2 c1=0 c0=0',
    'New Load.load1 bus1=load.1.2.3 phases=3 conn=wye kv=12.47 kw=100 pf=0.95',
    'CalcVoltageBases',
    'Solve',
]
for command in commands:
    dss.Text.Command(command)
assert dss.Solution.Converged()
dss.Circuit.SetActiveBus('load')
direct_v_pu = float(dss.Bus.puVmagAngle()[0])

case = Case.model_validate({
    'meta': {'name': 'public_first_circuit', 'mode': 'demonstrator'},
    'network': {
        'kind': 'inline', 'frequency_hz': 60,
        'inline': {
            'buses': [
                {'name': 'source', 'kv': 12.47, 'phases': 3},
                {'name': 'load', 'kv': 12.47, 'phases': 3},
            ],
            'lines': [{
                'name': 'line1', 'from_bus': 'source', 'to_bus': 'load',
                'length_km': 1.0, 'r1_ohm_per_km': 0.2,
                'x1_ohm_per_km': 0.4, 'r0_ohm_per_km': 0.6,
                'x0_ohm_per_km': 1.2, 'b1_us_per_km': 0.0,
            }],
            'loads': [{'id': 'load1', 'bus': 'load', 'phases': 3, 'kw': 100.0, 'pf': 0.95}],
            'external_grids': [{
                'name': 'grid', 'bus': 'source', 'pu': 1.0,
                'angle_deg': 0.0, 'sk3_mva': 1000.0, 'x_r_ratio': 10.0,
            }],
        },
    },
    'study': {'type': 'load_flow'},
})
run = run_study(case)
cept_v_pu = run.result.load_flow.voltage('load', 1)
assert cept_v_pu is not None
assert run.verification['passed'] is True
difference_pu = abs(direct_v_pu - cept_v_pu)
print({'direct_v_pu': direct_v_pu, 'cept_v_pu': cept_v_pu, 'difference_pu': difference_pu})
assert difference_pu < 1e-4


The small difference is a representation/rounding difference between direct commands and CEPT's typed inline translation. It is compared with an explicit teaching tolerance; it is not silently presented as a project acceptance tolerance.